# 1.) Initialize Spark Session

In [1]:
import pyspark
from pyspark import SparkConf, SparkContext
from pyspark.sql import *
import datetime

spark.stop()
spark = SparkSession \
.builder \
.appName("Data Quality Framework") \
.config("spark.master", "yarn")\
.config("spark.deploy-mode", "client")\
.config("spark.executor.instances", "10")\
.config("spark.executor.cores", "4")\
.config("spark.executor.memory", "15g")\
.config("spark.driver.cores", "8")\
.config("spark.driver.memory", "6g")\
.config("spark.queue", "qra_q1.analyt_sq1")\
.config("hive.exec.dynamic.partition", "true")\
.config("hive.exec.dynamic.partition.mode", "nonstrict")\
.enableHiveSupport() \
.getOrCreate()

spark.sql("use data_quality")  # Change this as needed
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

2020-01-31 10:04:53


# 3. Run Validation Rules

* Update Session_ID(s) to identify current batch/partition of data to validate

In [2]:
# Specify configuration files and dataset/session_id
config_path = "/mapr/datalake/optum/optuminsight/opsirsch1/qra/data_quality_framework/config/"
datasets_config_file = "dataset_config_edps.csv"
rules_library_file = "rule_library.csv"
dataset_rules_config = "dataset_edps_rule_config.csv"

# List of Dataset Id's to validate
dataset_id_list = [7]  

# Update this for current batch of data
new_session_ids = ['01032020A2014N'] # NonUHC Claims Year 2014

* Load the framework code by running cell labelled "2.) Load Code Framework" below.
* Then run below cell once to create validation output.

In [ ]:
validation_run_id = None
for session_id in new_session_ids:

    validation_run_id = load_and_run_multiple(
                                                config_path, 
                                                datasets_config_file,
                                                rules_library_file,
                                                dataset_rules_config,
                                                dataset_id_list,
                                                session_id,
                                                True
                                            )
print("Done.")

# 2.) Load Code Framework

In [4]:
from pyspark.sql.functions import col, lit, udf, desc, to_date, date_format, expr, count, when, countDistinct, sum
import time
from pyspark.sql.types import BooleanType, IntegerType, ArrayType, StringType
import json
import pprint
import datetime
import pandas as pd
import csv
from pyspark.sql.window import *
from pyspark.sql.functions import row_number
from pyspark.sql import Row
from collections import OrderedDict

# UDFS

spark.udf.register("count_distinct_chars_udf", lambda c: len(set(c)), IntegerType())

# METHODS

def load_datasets_config(config_path, config_file_name):
    dataset_config_file = config_path + config_file_name
    datasets_config = pd.read_csv(dataset_config_file, header=0, sep="|", keep_default_na=False).to_dict(orient='records')
    return datasets_config


def load_validation_rules(file_name):
    rule_library_file = config_path + file_name
    rules_lib = pd.read_csv(rule_library_file, header=0, sep="|", keep_default_na=False)
    return rules_lib


def load_lookup_csv(file_name):
    lookup_file = file_name
    lookup_data = pd.read_csv(lookup_file, header=0, sep="|", keep_default_na=False)
    return spark.createDataFrame(lookup_data).persist()


def load_dataset_rule_config(rules_lib, file_name):
    rule_config_file = config_path + file_name
    rules_config = pd.read_csv(rule_config_file, header=0, sep="|", keep_default_na=False)
    dataset_rules_config = pd.merge(rules_lib, rules_config, on='rule_id', how='inner').to_dict(orient='records')
    for r in dataset_rules_config:
        if "=" in r["rule_params"]:
            token = r["rule_params"][0:r["rule_params"].index("=")]
            val = r["rule_params"][r["rule_params"].index("=")+1:]
            r["descr"] = r["descr"].replace("{" + token + "}",val)
            r["expression"] = r["expression"].replace("{" + token + "}",val)
        elif "=" in r["default_params"]:
            for p in r["default_params"].split(","):
                token = p[0:p.index("=")]
                val = p[p.index("=")+1:]
                r["expression"] = r["expression"].replace("{" + token + "}",val)
                r["descr"] = r["descr"].replace("{" + token + "}",val)
    return dataset_rules_config


def get_dataset_config(id, datasets_config):
    return filter(lambda x: x["dataset_id"] == id, datasets_config)[0]


def load_dataset(dataset_config, partition_value=None, row_limit=0):
    if dataset_config['type'].lower() == "parquet":
        parquet_file_path = dataset_config['location']
        print("Loading " + parquet_file_path)
        df = spark.read.parquet(parquet_file_path)
    else:
        if dataset_config['query'].strip() == "":
            query = "select * from " + dataset_config['location'] 
            if dataset_config['partition_id_column_name'] != "":
                query += " where " + dataset_config['partition_id_column_name'] + " = '" + \
                (partition_value or dataset_config['partition_id_column_value']) + "'"
        else:
            query = dataset_config['query'].replace('#partition_id_column_value#',partition_value)
        print("Loading Dataset: " + dataset_config["descr"] + " (" + query + ")") 
        df = spark.sql(query)
    
    if row_limit==0:
        df = df.persist()
    else:
        df = df.limit(row_limit).persist() 
    df.registerTempTable("dataset_df_tmp")
    dataset_config['record_count'] = df.count()
    dataset_config['partition_id_column_value'] = partition_value or dataset_config['partition_id_column_value']
    print("Shape: " + str(dataset_config['record_count']) + " x " + str(len(df.columns)))
    print("")
    return df


def get_validation_run_id():
    return int(round(time.time()))


def dict_to_df(d):
    l = []
    l.append(d)
    df = spark.createDataFrame(Row(**x) for x in l)
    return df


def display_rule_mappings_per_column(dataset_df, ds_config, rules):
    cols = OrderedDict()
    for c1 in dataset_df.columns:
        cols[c1] = []
    cols[""] = []
    for r in rules:
        for c2 in r['columns'].split(','):
            cols[c2.strip()].append('Rule ' + str(r['rule_id']) + ": " + r['descr'])

    print("Dataset ID: " + str(ds_config["dataset_id"]))
    print("Description: " + ds_config["descr"])
    print("Location: " + ds_config["location"])
    print("Partition: " + ds_config["partition_id_column_name"] + "=" + ds_config["partition_id_value"])
    print("Record Count: " + str(ds_config["record_count"]))
    print("")

    for c in cols:
        if c=="":
            print("\nRow-based Rules")
        else:
            print(c)
        if len(cols[c]) == 0:
            print("\t(None)")
        else:
            for r in cols[c]:
                print("\t" + r)
        print("")


def create_validation_run_record(dataset_config, validation_run_id=None):
    if validation_run_id == None:
        validation_run_id = get_validation_run_id()
    validation_run = {}
    validation_run["validation_run_id"] = validation_run_id
    validation_run["rundate"] = datetime.datetime.now()
    validation_run["dataset_id"] = dataset_config['dataset_id']
    validation_run["dataset_descr"] = dataset_config['descr']
    validation_run["partition_id_column_name"] = dataset_config["partition_id_column_name"]
    validation_run["partition_id_column_value"] = dataset_config["partition_id_column_value"]
    validation_run["groupby_column_name"] = dataset_config["groupby_column_name"]
    validation_run["rundate_column_name"] = dataset_config["rundate_column_name"]
    return validation_run


def create_result_series(validation_run, rule, column, cnt, pct, total_cnt):
    s = pd.Series(data={'groupby_column_value': "",
                        'groupby_column_name': "",
                        'rule_id': int(rule['rule_id']),
                        'validation_run_id': int(validation_run["validation_run_id"]),
                        'rule_desc': rule['descr'],
                        'threshold_pct1': rule['threshold_pct1'],
                        'threshold_pct2': rule['threshold_pct2'],
                        'column_name': column,
                        'count': int(cnt),
                        'total_cnt': int(total_cnt),
                        'pct' : pct,
                        'dataset_id' : int(validation_run["dataset_id"])
                  })
    return s


def create_result_df(df, counts_df, validation_run, rule, column):
    groupby_column_name = validation_run["groupby_column_name"]
    df1 = df.withColumnRenamed(groupby_column_name,'groupby_column_value')\
        .withColumn('groupby_column_name', lit(groupby_column_name))\
        .withColumn('rule_id', lit(rule["rule_id"]))\
        .withColumn('validation_run_id', lit(validation_run["validation_run_id"]))\
        .withColumn('rule_desc', lit(rule["descr"]))\
        .withColumn('threshold_pct1', lit(rule["threshold_pct1"]))\
        .withColumn('threshold_pct2', lit(rule["threshold_pct2"]))\
        .withColumn('column_name', lit(column))\
        .withColumn('dataset_id', lit(validation_run["dataset_id"]))
    df1.registerTempTable("df1_tmp")
    counts_df.registerTempTable("counts_df_tmp")
    out_df = spark.sql("""
        SELECT a.*,
               a.count / b.count as pct,
               b.count as total_cnt
        FROM df1_tmp a
        LEFT JOIN counts_df_tmp b
            ON a.groupby_column_value = b.""" + groupby_column_name + """
    """)
    return out_df.toPandas()
 
    
def run_validation_rules(rules, dataset_df, dataset_config, validation_run_id=None):
    validation_run = create_validation_run_record(dataset_config, validation_run_id)
    run_df = dict_to_df(validation_run)
    if len(validation_run["groupby_column_name"]) > 1:
        out_df = run_validation_rules_with_groupby(validation_run, rules, dataset_df, dataset_config)
    else:
        out_df = run_validation_rules_without_groupby(validation_run, rules, dataset_df, dataset_config)
    return run_df, out_df
    
    
def run_validation_rules_without_groupby(validation_run, rules, dataset_df, dataset_config):

    out_df = pd.DataFrame()
    print("Processing rules..")
    
    # General Metrics
    cnt = dataset_df.count()
    total_cnt = cnt # used later
    pct = 1
    r = {}
    r["descr"] = "Total Record Count"
    r["rule_id"] = 0
    r["threshold_pct1"] = 100
    r["threshold_pct2"] = 100
    s = create_result_series(validation_run, r, "", cnt, pct, total_cnt)
    out_df = out_df.append(s, ignore_index=True)
    
    for r in rules:
        # Basic/Column Rules
        if r['dataset_id'] != dataset_config['dataset_id']: continue
        if r['type']=='Column':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            for c in r['columns'].split(','):
                cnt = dataset_df.select(lit(1)).where(expr(r['expression'].replace("{col}",c))).count()
                pct = float(cnt) / total_cnt
                s = create_result_series(validation_run, r, c, cnt, pct, total_cnt)
                out_df = out_df.append(s, ignore_index=True)
                
        if r['type']=='Lookup':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            for c in r['columns'].split(','):
                lookup_config_json = json.loads(r['expression'])
                lookup_df = load_lookup_csv(lookup_config_json["location"])
                lookup_col = lookup_config_json["column"]
                cnt = dataset_df\
                        .join(lookup_df, dataset_df[c] == lookup_df[lookup_col], how='left')\
                        .filter(col(lookup_col).isNull())\
                        .select(lit(1))\
                        .count()
                pct = float(cnt) / total_cnt
                s = create_result_series(validation_run, r, c, cnt, pct, total_cnt)
                out_df = out_df.append(s, ignore_index=True)
                               
        elif r['type']=='Row':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'])
            cnt = dataset_df.select(lit(1)).where(expr(r['expression'].replace("{col}",c))).count()
            pct = float(cnt) / total_cnt
            s = create_result_series(validation_run, r, "", cnt, pct, total_cnt)
            out_df = out_df.append(s, ignore_index=True)
        
        elif r['type']=='Distinct':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            for c in r['columns'].split(','):
                cnt = dataset_df.select([countDistinct(expr("{col}".replace("{col}",c)),c)]).collect()[0][0]
                pct = float(cnt) / total_cnt
                s = create_result_series(validation_run, r, c, cnt, pct, total_cnt)
                out_df = out_df.append(s, ignore_index=True)
        
        elif r['type']=='Duplicates':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            if r['columns'] == " " or r['columns'] == "*":
                c = dataset_df.columns
            else:
                c = r['columns']
            cnt = dataset_df.groupBy(c)\
                    .count()\
                    .where(col('count') > 1)\
                    .select(sum('count'))\
                    .collect()[0][0]
            if cnt is None: cnt = 0
            pct = float(cnt) / total_cnt
            s = create_result_series(validation_run, r, c, cnt, pct, total_cnt)
            out_df = out_df.append(s, ignore_index=True)
    
    return out_df


def run_validation_rules_with_groupby(validation_run, rules, dataset_df, dataset_config):

    out_df = pd.DataFrame()
    groupby_column_name = validation_run["groupby_column_name"]
    print("Processing rules..")
    
    # General Metrics
    r = {}
    r["descr"] = "Total Record Count"
    r["rule_id"] = 0
    r["threshold_pct1"] = 100
    r["threshold_pct2"] = 100
    print('Rule ' + str(r['rule_id']) + ": " + r['descr'])
    total_counts_df = dataset_df.groupBy(groupby_column_name).count()
    total_counts_df.show()
    total_counts_ldf = create_result_df(total_counts_df, total_counts_df, validation_run, r, "")
    out_df = out_df.append(total_counts_ldf, ignore_index=True)
    
    for r in rules:
        if r['dataset_id'] != dataset_config['dataset_id']: continue
        # Basic/Column Rules
        if r['type']=='Column':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            for c in r['columns'].split(','):
                #df = dataset_df.where(expr(r['expression'].replace("{col}",c))).groupBy(groupby_column_name).count()
                df = dataset_df.groupBy(groupby_column_name)\
                    .agg(sum(when(expr(r['expression'].replace("{col}",c)),1).otherwise(0)).alias("count"))
                df.show()
                ldf = create_result_df(df, total_counts_df, validation_run, r, c)
                out_df = out_df.append(ldf, ignore_index=True)
                
        if r['type']=='Row':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'])
            df = dataset_df.groupBy(groupby_column_name)\
                    .agg(sum(when(expr(r['expression'].replace("{col}",c)),1).otherwise(0)).alias("count"))
            ldf = create_result_df(df, total_counts_df, validation_run, r, c)
            out_df = out_df.append(ldf, ignore_index=True)
                
        if r['type']=='Lookup':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            for c in r['columns'].split(','):
                lookup_config_json = json.loads(r['expression'])
                lookup_df = load_lookup_csv(lookup_config_json["location"])
                lookup_col = lookup_config_json["column"]
                df = dataset_df\
                        .join(lookup_df, dataset_df[c] == lookup_df[lookup_col], how='left')\
                        .filter(col(lookup_col).isNull())\
                        .groupBy(groupby_column_name)\
                        .count()
                ldf = create_result_df(df, total_counts_df, validation_run, r, c)
                out_df = out_df.append(ldf, ignore_index=True)
      
        elif r['type']=='Distinct':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            for c in r['columns'].split(','):
                df = dataset_df.groupby(groupby_column_name).agg(countDistinct(c).alias("count"))
                ldf = create_result_df(df, total_counts_df, validation_run, r, c)
                out_df = out_df.append(ldf, ignore_index=True)
        
        elif r['type']=='Duplicates':
            print('Rule ' + str(r['rule_id']) + ": " + r['descr'] + " [" + r['columns'] + "]")
            if r['columns'] == " " or r['columns'] == "*":
                c = dataset_df.columns
            else:
                c = r['columns']
            df1 = dataset_df.withColumn("row_num", row_number().over(Window.partitionBy(c)\
                                                                     .orderBy(dataset_df.columns[0])))
            df1.registerTempTable("df1_tmp")
            df2 = spark.sql("""
                    SELECT """ + groupby_column_name + """, MAX(row_num)-1 as count
                    FROM df1_tmp
                    GROUP BY """ + groupby_column_name)
            df2.show()
            ldf = create_result_df(df2, total_counts_df, validation_run, r, "")
            out_df = out_df.append(ldf, ignore_index=True)

    return out_df


def save_output_to_hive(run_df, out_ldf, save_to_hive=False):   
    run_df.registerTempTable("run_df_tmp")
    if save_to_hive:
        spark.sql("""
            INSERT INTO dvf_validation_run
            SELECT validation_run_id,
                rundate,
                dataset_id,
                dataset_descr,
                partition_id_column_name,
                partition_id_column_value,
                groupby_column_name,
                rundate_column_name
            FROM run_df_tmp
        """)

    
    out_ldf[['groupby_column_name', 'groupby_column_value', 'column_name']] = out_ldf[['groupby_column_name', 'groupby_column_value', 'column_name']].astype(str)
    
    out_df = spark.createDataFrame(out_ldf)
    out_df.registerTempTable("out_df_tmp")
    out_df.show()
    if save_to_hive:
        spark.sql("""
            INSERT INTO dvf_validation_counts
            PARTITION(validation_run_id)
            SELECT rule_id,
                rule_desc,
                column_name,
                count as cnt,
                total_cnt,
                pct,
                threshold_pct1,
                threshold_pct2,
                groupby_column_name,
                groupby_column_value,
                dataset_id,
                validation_run_id   
            FROM out_df_tmp
        """)        

def load_and_run_multiple(config_path, 
                    datasets_config_file,
                    rules_library_file,
                    dataset_rules_config,
                    dataset_id_list,
                    partition_value,
                    save_to_hive = False,
                    validation_run_id = None,
                    row_limit = 0):
    
    datasets_config = load_datasets_config(config_path, datasets_config_file)
    rules_lib = load_validation_rules(rules_library_file)
    rules = load_dataset_rule_config(rules_lib, dataset_rules_config)
    
    for dataset_id in dataset_id_list:
        dataset_config = get_dataset_config(dataset_id, datasets_config)
        dataset_df = load_dataset(dataset_config, 
                                  partition_value=partition_value,
                                  row_limit=row_limit)
        run_df, out_ldf = run_validation_rules(rules, dataset_df, dataset_config, validation_run_id)
        save_output_to_hive(run_df, out_ldf, save_to_hive)
        validation_run_id = run_df.select("validation_run_id").collect()[0][0]
        print('\n-----------------------------\n')
        print(validation_run_id)
        print('\n-----------------------------\n')
    return validation_run_id



## (Optional) Code Extensions Specific to EDPS Reporting

In [18]:
# EDPS

def populate_dvf_edps_reporting():

    col_attrib_csv = load_lookup_csv("/home/nbetz1/projects/dvf/config/edps/dataset_column_attributes_edps.csv")
    col_attrib_csv.registerTempTable("col_attrib_csv_tmp")
    #col_attrib_csv.show(50,False)

    edps_results_df = spark.sql("""
        select v.dataset_id, 
            v.rundate, 
            v.partition_id_column_value as session_id,  
            v.dataset_descr, 
            case 
                when v.partition_id_column_value like '%U' then 'uhc'  
                else 'nonuhc' 
            end as uhcnonuhc, 
            case 
                when v.partition_id_column_value like '%2014U' then 2014 
                when v.partition_id_column_value like '%2014N' then 2014 
                when v.partition_id_column_value like '%2018U' then 2018
                when v.partition_id_column_value like '%2018N' then 2018
                when v.partition_id_column_value like '%2019U' then 2019
                when v.partition_id_column_value like '%2019N' then 2019
                when v.partition_id_column_value like '%2020U' then 2020
                when v.partition_id_column_value like '%2020N' then 2020
            end as claim_year, 
            v.rule_id, 
            v.rule_desc, 
            trim(v.column_name) as column_name, 
            v.cnt, 
            v.total_cnt, 
            v.pct, 
            v.validation_run_id,
            v.threshold_pct1,
            v.threshold_pct2,
            trim(c.description) as column_description
        from vw_dvf_validation v
        left join col_attrib_csv_tmp c
            on c.dataset_id = v.dataset_id
            and c.column_name = trim(v.column_name)
        join edps_run_ids_df_tmp e
            on e.validation_run_id = v.validation_run_id
            and e.dataset_id = v.dataset_id
        order by v.partition_id_column_value, v.dataset_id, v.rule_id
    """)
    edps_results_df.persist()
    edps_results_df.show(500,False)

    if False:
        edps_results_df.registerTempTable("edps_results_df_tmp")
        spark.sql("DROP TABLE IF EXISTS dvf_edps_reporting").show()
        spark.sql("""
            CREATE TABLE dvf_edps_reporting
            AS
            SELECT * from edps_results_df_tmp    
        """)

def get_latest_validation_runs():
    #TODO: Use new run_info table
    edps_run_ids_df = spark.sql("""
        select distinct
            partition_id_column_value as session_id, 
            dataset_id,
            dataset_descr,
            validation_run_id, 
            rundate,
            row_number() over (partition by partition_id_column_value, dataset_id order by validation_run_id desc) as rn
        from dvf_validation_run v
        where dataset_id in ({dataset_id_list})
            and partition_id_column_value IN ('{session_ids}')
        order by session_id desc, dataset_id
    """.format(session_ids = ",".join([str(elem) for elem in new_session_ids]), 
               dataset_id_list = ",".join([str(elem) for elem in dataset_id_list]),
               run_date_list = "'2020-01-09'"))\
    .filter("rn = 1")
    edps_run_ids_df.persist()
    edps_run_ids_df.registerTempTable("edps_run_ids_df_tmp")
    edps_run_ids_df.show(50)

def edps_populate_subclisk_counts(session_ids):

    sql = """
        SELECT session_id,sub_cli_sk, count(1) as cnt
        FROM EDPS_PROD.clm_header
        WHERE session_id IN ('""" + "','".join(session_ids) + """')
        GROUP BY session_id,sub_cli_sk
        ORDER BY session_id,sub_cli_sk
    """
    print(sql)
    sub_cli_sk_counts_df = spark.sql(sql)
    sub_cli_sk_counts_df.persist()
    sub_cli_sk_counts_df.show(200)

    sub_cli_sk_counts_df.registerTempTable("sub_cli_sk_counts_df_tmp")
    spark.sql("DROP TABLE IF EXISTS dvf_edps_reporting_subclisk_cnt").show()
    spark.sql("""
        CREATE TABLE dvf_edps_reporting_subclisk_cnt
        AS
        SELECT * from sub_cli_sk_counts_df_tmp    
    """).show()

def edps_get_high_level_counts(session_ids):
    counts_query = ("""
        select partition_id_column_value as session_id, 
            dataset_descr, 
            case 
                when partition_id_column_value like '%U' then 'uhc'  
                else 'nonuhc' 
            end as uhcnonuhc, 
            case 
                when partition_id_column_value like '%2014U' then 2014 
                when partition_id_column_value like '%2014N' then 2014 
            end as claim_year, 
            sum(distinct cnt) as Total_Count
        from vw_dvf_validation
        where partition_id_column_value in ('""" + "','".join(session_ids) + """')
            and rule_desc = 'Total Record Count'
        group by partition_id_column_value, 
            dataset_descr, 
            case 
                when partition_id_column_value like '%U' then 'uhc'  
                else 'nonuhc' 
            end , 
            case 
                when partition_id_column_value like '%2014U' then 2014 
                when partition_id_column_value like '%2014N' then 2014 
            end
        order by session_id, dataset_descr
    """)
    #print(counts_query)
    counts_df = spark.sql(counts_query)
    return counts_df

In [ ]:
get_latest_validation_runs()

In [ ]:
populate_dvf_edps_reporting()

# Below is only Dev/Test Code

In [9]:
config_path = "/mapr/datalake/optum/optuminsight/opsirsch1/qra/data_quality_framework/config/"
datasets_config = load_datasets_config(config_path, "dataset_config_edps.csv")
datasets_config

[{'dataset_id': 1,
  'descr': 'All Providers, GroupBy client_id',
  'groupby_column_name': 'client_id',
  'location': 'informatics_dev.all_providers',
  'partition_id_column_name': 'session_id',
  'partition_id_column_value': '120319064922695678435',
  'query': ' ',
  'rundate_column_name': 'run_month',
  'type': 'Hive'},
 {'dataset_id': 2,
  'descr': 'All Members',
  'groupby_column_name': ' ',
  'location': 'informatics_dev.all_members',
  'partition_id_column_name': 'session_id',
  'partition_id_column_value': '120319064922695678435',
  'query': ' ',
  'rundate_column_name': '',
  'type': 'Hive'},
 {'dataset_id': 3,
  'descr': 'All Providers,No GroupBy',
  'groupby_column_name': '',
  'location': 'informatics_dev.all_providers',
  'partition_id_column_name': 'session_id',
  'partition_id_column_value': '120319064922695678435',
  'query': ' ',
  'rundate_column_name': 'run_month',
  'type': 'Hive'},
 {'dataset_id': 4,
  'descr': 'All Providers,No GroupBy, Filtered',
  'groupby_column

### Test Load Rules Library

In [10]:
rules_lib = load_validation_rules("rule_library.csv")
rules_lib

,rule_id,descr,type,expression,default_params
0,1,"Column Is Null or Empty or ""unknown""",Column,"coalesce({col},'') = '' or upper({col}) = 'UNK...",
1,2,"Column Is Null or Empty, but {col2} Has a Value",Column,"(coalesce({col},'') = '' or upper({col}) = 'UN...",col2=SSN
2,3,Distinct values,Distinct,,
3,4,Duplicate values,Duplicates,,
4,5,Column Is Numeric,Column,{col} rlike '^[0-9]+',
5,6,Date column is equal to {date},Column,{col} == '{date}',date=12/31/2199
6,7,Column contains special characters,Column,{col} not rlike '[a-zA-Z0-9]',
7,8,Column contains special characters other than ...,Column,{col} not rlike '[a-zA-Z\s]',
8,9,Column contains non-numeric characters,Column,{col} not rlike '^[0-9]+',
9,10,"Street address columns contain ""suite""",Row,"upper(concat(street1,street2)) rlike 'SUITE'",


###  Test Load Dataset-Rule Configuration

In [11]:
rules = load_dataset_rule_config(rules_lib, "dataset_edps_rule_config.csv")
rules

[{'columns': '*',
  'dataset_id': 7,
  'default_params': ' ',
  'descr': 'Duplicate values',
  'expression': ' ',
  'rule_id': 4,
  'rule_params': ' ',
  'threshold_pct1': 100.0,
  'threshold_pct2': 100.0,
  'type': 'Duplicates'},
 {'columns': '*',
  'dataset_id': 8,
  'default_params': ' ',
  'descr': 'Duplicate values',
  'expression': ' ',
  'rule_id': 4,
  'rule_params': ' ',
  'threshold_pct1': 100.0,
  'threshold_pct2': 100.0,
  'type': 'Duplicates'},
 {'columns': '*',
  'dataset_id': 9,
  'default_params': ' ',
  'descr': 'Duplicate values',
  'expression': ' ',
  'rule_id': 4,
  'rule_params': ' ',
  'threshold_pct1': 100.0,
  'threshold_pct2': 100.0,
  'type': 'Duplicates'},
 {'columns': '*',
  'dataset_id': 10,
  'default_params': ' ',
  'descr': 'Duplicate values',
  'expression': ' ',
  'rule_id': 4,
  'rule_params': ' ',
  'threshold_pct1': 100.0,
  'threshold_pct2': 100.0,
  'type': 'Duplicates'},
 {'columns': 'batch_id, claim_from_date, claim_id, claim_to_date, claim_typ

### Test Load Dataset

In [14]:
dataset_config = get_dataset_config(7, datasets_config)
dataset_df = load_dataset(dataset_config, 
                          new_session_ids[0],
                          row_limit=10000)
dataset_df.printSchema()

Loading Dataset: clm_header (select * from edps_prod.clm_header where session_id = '01032020A2014N')
Shape: 10000 x 25

root
 |-- med_clm_sk: integer (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- place_of_service_code: string (nullable = true)
 |-- claim_facility_cd: string (nullable = true)
 |-- claim_from_date: date (nullable = true)
 |-- claim_to_date: date (nullable = true)
 |-- claim_admissing_date: date (nullable = true)
 |-- claim_discharge_date: date (nullable = true)
 |-- edps_src_sys_clm_id: string (nullable = true)
 |-- hic_number: string (nullable = true)
 |-- claim_type_code: string (nullable = true)
 |-- bill_type_code: string (nullable = true)
 |-- qual_flag: string (nullable = true)
 |-- latest_hic_mbi: string (nullable = true)
 |-- edps_clm_med_rec_num: string (nullable = true)
 |-- asm_ind: string (nullable = true)
 |-- asm_suppl_typ: string (nullable = true)
 |-- asm_suppl_typ_desc: string (nullable = true)
 |-- clrhse_clm_ctl_num: string (nullable =

##### Dev/Test a rule to validate an all numeric value with a regular expression

                
Group By Count of Values  
df.groupBy(c).count().where(expr(test_expr.replace("{col}",c))).show(5)  

All Values  
df.select(c).alias(c).where(expr(test_expr.replace("{col}",c))).show(5)  

Distinct Values  
df.select(c).where(expr(test_expr.replace("{col}",c))).distinct().show(5)  

In [9]:
e = "{col} rlike '^[0-9]+'"  #The Expression we want to test
c = "phone_number"           #The column name
test_df = spark.sql("select '1111111' as phone_number")  # Mock data value for testing
test_df.show()
test_df.select(c).where(expr(e.replace("{col}",c))).count()

+------------+
|phone_number|
+------------+
|     1111111|
+------------+



1

In [3]:
test_df = spark.sql("select '1111111' as phone_number")
test_df.show()
test_df.select(lit(1)).where(count_distinct_chars("phone_number") == 1).count()

+------------+
|phone_number|
+------------+
|     1111111|
+------------+



1

In [38]:
spark.stop()